# LeetCode 1428 — Leftmost Column with at Least a One bằng Python

Link đề: https://leetcode.com/problems/leftmost-column-with-at-least-a-one/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Matrix, Interactive

> **Ghi chú Premium:** đây là bài **LeetCode Premium** (🔒). Đề bài và chữ ký trong notebook được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1428/). Nếu có tài khoản Premium, nên đối chiếu lại với đề gốc.
>
> Đây là bài **tương tác (interactive)**: LeetCode cung cấp đối tượng `BinaryMatrix`. Trong notebook, lớp `BinaryMatrix` là **bản giả lập** do ta tự viết (có đếm số lần gọi `get` và giới hạn 1000 lần) để chạy thử.

Skeleton LeetCode (theo doocs):

```python
# """
# This is BinaryMatrix's API interface.
# You should not implement it, or speculate about its implementation
# """
# class BinaryMatrix(object):
#    def get(self, row: int, col: int) -> int:
#    def dimensions(self) -> list[]:

class Solution:
    def leftMostColumnWithOne(self, binaryMatrix: "BinaryMatrix") -> int:
        ...
```

## 1. Mục tiêu học

1. Làm quen với **bài tương tác**: không đọc trực tiếp dữ liệu, chỉ hỏi qua API (`get`, `dimensions`) và bị **giới hạn số lần hỏi** — độ phức tạp đo bằng **số lần gọi**.
2. Khai thác tính chất **mỗi hàng đã sắp xếp** (0…0 1…1): vị trí số 1 đầu tiên của hàng tìm được bằng **binary search**.
3. Biết kỹ thuật **đi bậc thang** (staircase) từ góc trên-phải: mỗi lần hỏi loại bỏ hẳn một hàng hoặc một cột ⇒ `O(R + C)` lần gọi.
4. Nhận ra cách **binary search trên đáp án** (cột) với vị từ đơn điệu "cột `c` có số 1".
5. Tự dựng **bộ giả lập** có đếm lời gọi để kiểm thử, và tránh các lỗi: quét tuyến tính vượt giới hạn, đi sai hướng, nhầm `rows/cols`.

## 2. Đề bài & ví dụ

Một **ma trận nhị phân sắp theo hàng** (row-sorted binary matrix) là ma trận chỉ gồm `0` và `1`, mỗi hàng **không giảm** (mọi số 0 đứng trước mọi số 1).

Cho ma trận như vậy `binaryMatrix`, trả về **chỉ số cột (từ 0) nhỏ nhất** có chứa ít nhất một số `1`. Nếu không có cột nào như vậy, trả `-1`.

Bạn **không** truy cập trực tiếp ma trận, chỉ qua giao diện:
- `BinaryMatrix.get(row, col)` — trả phần tử tại `(row, col)` (chỉ số từ 0);
- `BinaryMatrix.dimensions()` — trả `[rows, cols]`.

Bài nộp gọi `get` **quá 1000 lần** sẽ bị chấm **Wrong Answer**. (Khi tự test, input là toàn bộ ma trận `mat`.)

**Ví dụ 1:** `mat = [[0,0],[1,1]]` → `0`.

**Ví dụ 2:** `mat = [[0,0],[0,1]]` → `1`.

**Ví dụ 3:** `mat = [[0,0],[0,0]]` → `-1`.

**Ràng buộc:** `rows == mat.length`, `cols == mat[i].length`, `1 <= rows, cols <= 100`, `mat[i][j]` là `0` hoặc `1`, mỗi `mat[i]` không giảm.

In [1]:
import random
import time
from bisect import bisect_left
from typing import List

class TooManyCalls(Exception):
    pass

class BinaryMatrix:
    """BẢN GIẢ LẬP giao diện BinaryMatrix của LeetCode (không phải code của LeetCode).
    - get(row, col): trả mat[row][col], đếm số lần gọi; vượt LIMIT thì ném TooManyCalls.
    - dimensions(): trả [rows, cols]."""
    LIMIT = 1000

    def __init__(self, mat, strict=True):
        self._mat = [row[:] for row in mat]
        self.calls = 0
        self.strict = strict          # strict=False: chỉ đếm, không ném lỗi (để đo)

    def get(self, row: int, col: int) -> int:
        self.calls += 1
        if self.strict and self.calls > self.LIMIT:
            raise TooManyCalls(f"gọi get quá {self.LIMIT} lần")
        return self._mat[row][col]

    def dimensions(self) -> list:
        return [len(self._mat), len(self._mat[0])]

def run(solver, mat, strict=True):
    """Bộ chạy kiểu LeetCode: trả (đáp án, số lần gọi get, trạng thái)."""
    bm = BinaryMatrix(mat, strict)
    try:
        ans = solver(bm)
    except TooManyCalls:
        return None, bm.calls, "Wrong Answer (quá 1000 lần gọi get)"
    status = "OK" if bm.calls <= BinaryMatrix.LIMIT else "Wrong Answer (quá 1000 lần gọi get)"
    return ans, bm.calls, status

def truth(mat):
    """Đáp án chuẩn tính trực tiếp từ ma trận (chỉ dùng để kiểm thử)."""
    cols = [j for j in range(len(mat[0])) if any(row[j] for row in mat)]
    return min(cols) if cols else -1

examples = [([[0, 0], [1, 1]], 0), ([[0, 0], [0, 1]], 1), ([[0, 0], [0, 0]], -1)]
for mat, want in examples:
    print(mat, "->", want, "| truth =", truth(mat))

[[0, 0], [1, 1]] -> 0 | truth = 0
[[0, 0], [0, 1]] -> 1 | truth = 1
[[0, 0], [0, 0]] -> -1 | truth = -1


## 3. Kiến thức nền

### 3.1 Bài tương tác & mô hình chi phí "số lần hỏi"

Ở bài thường, ta đếm số phép tính. Ở bài tương tác, **mỗi lần gọi `get` là một lần hỏi tốn kém** và bị giới hạn. Ma trận lớn nhất `100 × 100 = 10 000` ô ⇒ không được đọc hết; giới hạn 1000 buộc ta dùng cấu trúc "hàng đã sắp".

In [2]:
bm = BinaryMatrix([[0, 0, 1], [0, 1, 1]])
print(bm.dimensions(), bm.get(0, 2), bm.get(1, 0), "| số lần gọi:", bm.calls)

[2, 3] 1 0 | số lần gọi: 2


### 3.2 Binary search tìm vị trí đầu tiên thoả điều kiện

Trên một dãy đơn điệu `F F … F T T … T`, tìm chỉ số `T` đầu tiên trong `[lo, hi)`:

```python
while lo < hi:
    mid = (lo + hi) // 2
    if cond(mid): hi = mid
    else: lo = mid + 1
return lo     # = hi ban đầu nếu không có T
```

Số bước `⌈log2(n + 1)⌉` ⇒ với `n = 100` tối đa 7 lần hỏi.

In [3]:
row = [0, 0, 0, 1, 1, 1, 1]
lo, hi, steps = 0, len(row), 0
while lo < hi:
    mid = (lo + hi) // 2; steps += 1
    if row[mid] == 1: hi = mid
    else: lo = mid + 1
print("số 1 đầu tiên ở", lo, "sau", steps, "bước")

số 1 đầu tiên ở 3 sau 3 bước


### 3.3 `bisect_left` với `key` (Python ≥ 3.10)

`bisect_left(seq, x, key=f)` tìm vị trí đầu tiên mà `f(seq[i]) >= x`. Với `seq = range(n)` và `key=lambda j: bm.get(r, j)` ta binary search **trực tiếp trên API** mà không tạo list (lời giải doocs dùng cách này). `range` hỗ trợ truy cập theo chỉ số nên `bisect` dùng được.

In [4]:
bm = BinaryMatrix([[0, 0, 0, 1, 1]])
j = bisect_left(range(5), 1, key=lambda k: bm.get(0, k))
print("vị trí số 1 đầu tiên:", j, "| số lần gọi:", bm.calls)

vị trí số 1 đầu tiên: 3 | số lần gọi: 3


### 3.4 Hàm `min` với giá trị "vô cực" và quy ước `-1`

Ta thường khởi tạo `ans = cols` (một giá trị "không có") rồi `ans = min(ans, j)`; cuối cùng đổi `cols` thành `-1`. Tránh khởi tạo `ans = -1` rồi `min` (vì `-1` luôn nhỏ nhất!).

In [5]:
cols = 5
ans = cols
for j in [5, 3, 5]:          # 5 nghĩa là hàng không có số 1
    ans = min(ans, j)
print(-1 if ans == cols else ans)

3


## 4. Ý tưởng nền tảng / trực giác

Xét ma trận 4 × 5 (ta "nhìn trộm" để hiểu, thuật toán thật không được nhìn):

```
hàng 0: 0 0 0 1 1     -> số 1 đầu tiên ở cột 3
hàng 1: 0 0 1 1 1     -> cột 2
hàng 2: 0 0 0 0 0     -> không có
hàng 3: 0 1 1 1 1     -> cột 1
```

Đáp án = min(3, 2, 1) = **1**.

**Quan sát bậc thang:** đứng ở góc trên-phải `(0, 4)`.
- Thấy `1` ⇒ cột 4 có số 1 (ứng viên), và mọi ô bên **trái** có thể còn 1 ⇒ đi **trái**.
- Thấy `0` ⇒ vì hàng đã sắp, mọi ô bên trái trong hàng này cũng là 0 ⇒ hàng này hết hy vọng ⇒ đi **xuống**.

Mỗi bước loại một cột hoặc một hàng ⇒ tối đa `R + C` lần hỏi.

In [6]:
demo = [[0, 0, 0, 1, 1],
        [0, 0, 1, 1, 1],
        [0, 0, 0, 0, 0],
        [0, 1, 1, 1, 1]]
for r, row in enumerate(demo):
    first = row.index(1) if 1 in row else None
    print(f"hàng {r}: {row} -> số 1 đầu tiên: {first}")
print("đáp án:", truth(demo))

hàng 0: [0, 0, 0, 1, 1] -> số 1 đầu tiên: 3
hàng 1: [0, 0, 1, 1, 1] -> số 1 đầu tiên: 2
hàng 2: [0, 0, 0, 0, 0] -> số 1 đầu tiên: None
hàng 3: [0, 1, 1, 1, 1] -> số 1 đầu tiên: 1
đáp án: 1


## 5. Cách 1 — Brute Force / baseline: quét tuyến tính

**Ý tưởng:** duyệt **cột** từ trái sang phải, với mỗi cột duyệt mọi hàng; gặp số 1 đầu tiên thì trả về cột đó. (Hoặc mỗi hàng quét từ trái tới số 1 đầu tiên.) Đúng, nhưng trường hợp xấu đọc gần hết ma trận.

In [7]:
class SolutionLinear:
    def leftMostColumnWithOne(self, binaryMatrix: "BinaryMatrix") -> int:
        rows, cols = binaryMatrix.dimensions()
        for c in range(cols):
            for r in range(rows):
                if binaryMatrix.get(r, c) == 1:
                    return c
        return -1

lin = lambda bm: SolutionLinear().leftMostColumnWithOne(bm)
for mat, want in examples:
    print(run(lin, mat), "mong đợi", want)

(0, 2, 'OK') mong đợi 0
(1, 4, 'OK') mong đợi 1
(-1, 4, 'OK') mong đợi -1


**Độ phức tạp (số lần gọi):** `O(R · C)`, tệ nhất `10 000` lần với ma trận `100 × 100` toàn 0 (hoặc chỉ có 1 ở cột cuối) ⇒ **vượt giới hạn 1000** ⇒ Wrong Answer, dù đáp án đúng.

In [8]:
worst = [[0] * 100 for _ in range(100)]
worst[99][99] = 1
print("strict    :", run(lin, worst))
print("chỉ đếm   :", run(lin, worst, strict=False))

strict    : (None, 1001, 'Wrong Answer (quá 1000 lần gọi get)')
chỉ đếm   : (99, 10000, 'Wrong Answer (quá 1000 lần gọi get)')


**Nút thắt:** không tận dụng việc hàng đã sắp xếp. Một lần hỏi thấy `0` ở `(r, c)` cho biết **cả đoạn** `(r, 0..c)` là 0 — quét tuyến tính lãng phí thông tin này.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — mỗi hàng đơn điệu:** vị trí số 1 đầu tiên của hàng `r` là điểm chuyển `0 → 1` ⇒ **binary search** với `≤ ⌈log2(C+1)⌉ = 7` lần hỏi. Đáp án = min trên các hàng ⇒ `≤ 100 · 7 = 700` lần ⇒ đạt giới hạn.

**Insight 2 — bậc thang:** giữ con trỏ `(r, c)` bắt đầu `(0, C−1)`. Bất biến: *mọi cột `> c` đã biết là có 1 (hoặc ngoài cuộc), các hàng `< r` không còn số 1 nào ở cột `≤ c`*.
- **Nếu** `get(r, c) == 1` **thì** `ans = c`, `c −= 1` (thử cột trái).
- **Nếu** `get(r, c) == 0` **thì** `r += 1` (hàng này không có 1 ở cột `≤ c`).
Dừng khi `r == R` hoặc `c < 0`. Tối đa `R + C − 1 ≤ 199` lần hỏi.

**Insight 3 — binary search trên đáp án:** vị từ `P(c)` = "cột `c` có số 1" là **đơn điệu**: nếu cột `c` có 1 ở hàng `r` thì mọi cột `c' > c` cũng có 1 ở hàng `r`. ⇒ binary search `c` nhỏ nhất với `P(c)` đúng; mỗi lần kiểm tra tốn `R` lần hỏi ⇒ `R · ⌈log2(C+1)⌉ ≤ 700`.

**Insight 4 — tỉa binary search theo hàng:** khi đã có ứng viên `ans`, hàng sau chỉ cần tìm trong `[0, ans)`; nếu `get(r, ans−1) == 0` thì bỏ qua hàng luôn.

## 7. Các cách tiếp theo

### Cách 2 — Binary search trên từng hàng

**Từng đoạn:**
1. `rows, cols = dimensions()`; `ans = cols`.
2. Với mỗi hàng `r`: binary search trên `[0, cols)` tìm số 1 đầu tiên `lo` (bằng `cols` nếu không có).
3. `ans = min(ans, lo)`.
4. Trả `-1` nếu `ans == cols`.

In [9]:
class SolutionBinaryRows:
    def leftMostColumnWithOne(self, binaryMatrix: "BinaryMatrix") -> int:
        rows, cols = binaryMatrix.dimensions()
        ans = cols
        for r in range(rows):
            lo, hi = 0, cols
            while lo < hi:
                mid = (lo + hi) // 2
                if binaryMatrix.get(r, mid) == 1:
                    hi = mid
                else:
                    lo = mid + 1
            ans = min(ans, lo)
        return -1 if ans == cols else ans

brows = lambda bm: SolutionBinaryRows().leftMostColumnWithOne(bm)
for mat, want in examples:
    print(run(brows, mat), "mong đợi", want)
print("worst 100x100:", run(brows, worst))

(0, 3, 'OK') mong đợi 0
(1, 3, 'OK') mong đợi 1
(-1, 2, 'OK') mong đợi -1
worst 100x100: (99, 601, 'OK')


Phiên bản dùng `bisect_left(..., key=...)` như doocs (Python 3.10+):

In [10]:
class SolutionBisect:
    def leftMostColumnWithOne(self, binaryMatrix: "BinaryMatrix") -> int:
        m, n = binaryMatrix.dimensions()
        ans = n
        for i in range(m):
            j = bisect_left(range(n), 1, key=lambda k: binaryMatrix.get(i, k))
            ans = min(ans, j)
        return -1 if ans >= n else ans

bis = lambda bm: SolutionBisect().leftMostColumnWithOne(bm)
print([run(bis, mat)[0] for mat, _ in examples], run(bis, worst))

[0, 1, -1] (99, 601, 'OK')


**Độ phức tạp:** `R · ⌈log2(C + 1)⌉` lần gọi (≤ 700), thời gian tương ứng, `O(1)` bộ nhớ.

**Cải tiến nhỏ (Insight 4):** chỉ tìm trong `[0, ans)`; nếu `get(r, ans − 1) == 0` thì hàng này không cải thiện được, bỏ qua sau 1 lần hỏi.

In [11]:
class SolutionBinaryPruned:
    def leftMostColumnWithOne(self, binaryMatrix: "BinaryMatrix") -> int:
        rows, cols = binaryMatrix.dimensions()
        ans = cols
        for r in range(rows):
            if ans == 0:
                break
            if binaryMatrix.get(r, ans - 1) == 0:     # không thể có 1 ở cột < ans
                continue
            lo, hi = 0, ans - 1                       # biết chắc (r, ans-1) là 1
            while lo < hi:
                mid = (lo + hi) // 2
                if binaryMatrix.get(r, mid) == 1:
                    hi = mid
                else:
                    lo = mid + 1
            ans = lo
        return -1 if ans == cols else ans

bpr = lambda bm: SolutionBinaryPruned().leftMostColumnWithOne(bm)
print([run(bpr, mat)[0] for mat, _ in examples], run(bpr, worst), run(bpr, demo))

[0, 1, -1] (99, 106, 'OK') (1, 8, 'OK')


### Cách 3 — Đi bậc thang từ góc trên-phải (tối ưu)

**Từng đoạn:**
1. `r, c = 0, cols − 1`; `ans = -1`.
2. Khi `r < rows` và `c >= 0`: hỏi `get(r, c)`.
3. Bằng `1` ⇒ `ans = c; c −= 1`. Bằng `0` ⇒ `r += 1`.
4. Trả `ans`.

In [12]:
class SolutionStaircase:
    def leftMostColumnWithOne(self, binaryMatrix: "BinaryMatrix") -> int:
        rows, cols = binaryMatrix.dimensions()
        r, c = 0, cols - 1
        ans = -1
        while r < rows and c >= 0:
            if binaryMatrix.get(r, c) == 1:
                ans = c
                c -= 1
            else:
                r += 1
        return ans

stair = lambda bm: SolutionStaircase().leftMostColumnWithOne(bm)
for mat, want in examples:
    print(run(stair, mat), "mong đợi", want)
print("worst 100x100:", run(stair, worst), "| demo:", run(stair, demo))

(0, 3, 'OK') mong đợi 0
(1, 3, 'OK') mong đợi 1
(-1, 2, 'OK') mong đợi -1
worst 100x100: (99, 101, 'OK') | demo: (1, 8, 'OK')


**Độ phức tạp:** mỗi lần hỏi hoặc `r` tăng hoặc `c` giảm ⇒ tối đa `R + C` lần gọi (≤ 200), `O(1)` bộ nhớ.

### Cách 4 — Binary search trên **cột đáp án**

**Từng đoạn:**
1. `has_one(c)` = có hàng nào `get(r, c) == 1` (dừng sớm khi gặp).
2. Binary search `c ∈ [0, cols)` nhỏ nhất với `has_one(c)`; nếu không có thì `-1`.

Cách này thể hiện tư duy "**binary search trên đáp án**" — dùng được khi vị từ đơn điệu.

In [13]:
class SolutionColumnSearch:
    def leftMostColumnWithOne(self, binaryMatrix: "BinaryMatrix") -> int:
        rows, cols = binaryMatrix.dimensions()

        def has_one(c: int) -> bool:
            return any(binaryMatrix.get(r, c) == 1 for r in range(rows))

        lo, hi = 0, cols
        while lo < hi:
            mid = (lo + hi) // 2
            if has_one(mid):
                hi = mid
            else:
                lo = mid + 1
        return -1 if lo == cols else lo

colsrch = lambda bm: SolutionColumnSearch().leftMostColumnWithOne(bm)
print([run(colsrch, mat)[0] for mat, _ in examples], run(colsrch, worst), run(colsrch, demo))

[0, 1, -1] (99, 700, 'OK') (1, 10, 'OK')


**Độ phức tạp:** `⌈log2(C+1)⌉` lần kiểm tra × tối đa `R` lần gọi ⇒ ≤ 700 lần gọi. Kém cách 3 nhưng vẫn đạt; hữu ích khi nhớ pattern "binary search on answer".

## 8. TRACE từng bước

Trace cách 3 trên ma trận `demo` 4 × 5:

| bước | `(r, c)` | `get` | hành động | `ans` |
|---|---|---|---|---|
| 1 | (0,4) | 1 | trái | 4 |
| 2 | (0,3) | 1 | trái | 3 |
| 3 | (0,2) | 0 | xuống | 3 |
| 4 | (1,2) | 1 | trái | 2 |
| 5 | (1,1) | 0 | xuống | 2 |
| 6 | (2,1) | 0 | xuống | 2 |
| 7 | (3,1) | 1 | trái | 1 |
| 8 | (3,0) | 0 | xuống ⇒ `r = 4` dừng | 1 |

8 lần gọi `get`.

In [14]:
def trace_staircase(mat):
    bm = BinaryMatrix(mat)
    rows, cols = bm.dimensions()
    r, c, ans = 0, cols - 1, -1
    while r < rows and c >= 0:
        pr, pc = r, c
        v = bm.get(r, c)
        if v == 1:
            ans = c; act = "thấy 1 -> ans=c, sang TRÁI"
            c -= 1
        else:
            act = "thấy 0 -> xuống DƯỚI"
            r += 1
        print(f"lần gọi {bm.calls:2d}: get({pr},{pc})={v} | {act:27s} | r={r}, c={c}, ans={ans}")
    print("kết quả:", ans, "| tổng số lần gọi:", bm.calls)
    return ans

_ = trace_staircase(demo)

lần gọi  1: get(0,4)=1 | thấy 1 -> ans=c, sang TRÁI  | r=0, c=3, ans=4
lần gọi  2: get(0,3)=1 | thấy 1 -> ans=c, sang TRÁI  | r=0, c=2, ans=3
lần gọi  3: get(0,2)=0 | thấy 0 -> xuống DƯỚI        | r=1, c=2, ans=3
lần gọi  4: get(1,2)=1 | thấy 1 -> ans=c, sang TRÁI  | r=1, c=1, ans=2
lần gọi  5: get(1,1)=0 | thấy 0 -> xuống DƯỚI        | r=2, c=1, ans=2
lần gọi  6: get(2,1)=0 | thấy 0 -> xuống DƯỚI        | r=3, c=1, ans=2
lần gọi  7: get(3,1)=1 | thấy 1 -> ans=c, sang TRÁI  | r=3, c=0, ans=1
lần gọi  8: get(3,0)=0 | thấy 0 -> xuống DƯỚI        | r=4, c=0, ans=1
kết quả: 1 | tổng số lần gọi: 8


Trace cách 2 (binary search từng hàng) trên `demo` — in `lo, hi, mid` ở mỗi lần hỏi:

In [15]:
def trace_binary_rows(mat):
    bm = BinaryMatrix(mat)
    rows, cols = bm.dimensions()
    ans = cols
    for r in range(rows):
        lo, hi = 0, cols
        steps = []
        while lo < hi:
            mid = (lo + hi) // 2
            v = bm.get(r, mid)
            steps.append(f"mid={mid}->{v}")
            if v == 1: hi = mid
            else: lo = mid + 1
        ans = min(ans, lo)
        print(f"hàng {r}: " + ", ".join(steps) + f" => số 1 đầu tiên {lo if lo < cols else 'không có'}; ans={ans}")
    print("kết quả:", -1 if ans == cols else ans, "| số lần gọi:", bm.calls)

trace_binary_rows(demo)

hàng 0: mid=2->0, mid=4->1, mid=3->1 => số 1 đầu tiên 3; ans=3
hàng 1: mid=2->1, mid=1->0 => số 1 đầu tiên 2; ans=2
hàng 2: mid=2->0, mid=4->0 => số 1 đầu tiên không có; ans=2
hàng 3: mid=2->1, mid=1->1, mid=0->0 => số 1 đầu tiên 1; ans=1
kết quả: 1 | số lần gọi: 10


## 9. Code nộp LeetCode

Dùng cách 3 (bậc thang) — tối đa `R + C` lần gọi, đúng chữ ký theo doocs:

In [16]:
# """
# This is BinaryMatrix's API interface.
# You should not implement it, or speculate about its implementation
# """
# class BinaryMatrix(object):
#    def get(self, row: int, col: int) -> int:
#    def dimensions(self) -> list[]:

class Solution:
    def leftMostColumnWithOne(self, binaryMatrix: "BinaryMatrix") -> int:
        rows, cols = binaryMatrix.dimensions()
        r, c = 0, cols - 1          # bắt đầu ở góc trên-phải
        ans = -1
        while r < rows and c >= 0:
            if binaryMatrix.get(r, c) == 1:
                ans = c             # cột c có số 1, thử sang trái
                c -= 1
            else:
                r += 1              # hàng r không có 1 ở cột <= c
        return ans

sol = lambda bm: Solution().leftMostColumnWithOne(bm)
for mat, want in examples:
    a, calls, st = run(sol, mat)
    assert a == want and st == "OK"
print("3 ví dụ: OK")

3 ví dụ: OK


## 10. Vì sao thuật toán đúng

**Cách 3 (bậc thang).** Gọi `best` là đáp án đúng. Bất biến trước mỗi lần hỏi tại `(r, c)`:
1. Mọi hàng `r' < r` **không có số 1 ở cột ≤ c**.
2. Nếu `ans ≠ −1` thì cột `ans = c + 1` có số 1 (và mọi cột từ `c + 1` trở đi đã "xử lý xong").

Bước `get(r, c) = 1`: cột `c` có số 1 ⇒ `ans = c` hợp lệ; giảm `c` giữ (1) vì ràng buộc chỉ chặt hơn. Bước `get(r, c) = 0`: hàng `r` sắp xếp ⇒ `(r, 0..c)` toàn 0 ⇒ thêm hàng `r` vào tập ở (1).

Khi dừng: nếu `c < 0` thì `ans = 0` — nhỏ nhất có thể. Nếu `r = rows` thì theo (1) **không hàng nào** có số 1 ở cột `≤ c` ⇒ không cột nào `≤ c` chứa 1, còn cột `c + 1` (nếu `ans ≠ −1`) có 1 ⇒ `ans = best`. Nếu `ans = −1` thì `c = cols − 1` và không cột nào có 1 ⇒ `−1` đúng.

**Cách 2:** binary search trên dãy đơn điệu trả đúng điểm chuyển của từng hàng; min các hàng là cột nhỏ nhất có 1. **Cách 4:** vị từ "cột có 1" đơn điệu (lập luận ở mục 6) nên binary search đúng.

## 11. Phân tích Time / Space

Gọi `R` = số hàng, `C` = số cột (đều ≤ 100). Chi phí chính là **số lần gọi `get`**:

| Cách | Số lần gọi `get` (tệ nhất) | Với 100×100 | Space |
|---|---|---|---|
| 1. Quét tuyến tính | `R · C` | 10 000 ✘ | `O(1)` |
| 2. Binary search mỗi hàng | `R · ⌈log2(C+1)⌉` | 700 ✔ | `O(1)` |
| 2b. Binary search có tỉa | `≤ R + R·log C` (thường ít hơn nhiều) | ≤ 700 ✔ | `O(1)` |
| 3. Bậc thang | `R + C − 1` | 199 ✔ | `O(1)` |
| 4. Binary search trên cột | `R · ⌈log2(C+1)⌉` | 700 ✔ | `O(1)` |

Thời gian chạy tỉ lệ với số lần gọi.

In [17]:
rng = random.Random(1428)
def rand_mat(rng, R, C, p_empty=0.3):
    mat = []
    for _ in range(R):
        k = C if rng.random() < p_empty else rng.randint(0, C)   # k = vị trí số 1 đầu tiên
        mat.append([0] * k + [1] * (C - k))
    return mat

solvers = {"linear": lin, "binary": brows, "bisect": bis, "pruned": bpr, "stair": stair, "colsearch": colsrch}
mats = [worst, [[0] * 100 for _ in range(100)], [[1] * 100 for _ in range(100)], rand_mat(rng, 100, 100)]
names = ["1 ở góc dưới-phải", "toàn 0", "toàn 1", "ngẫu nhiên"]
for nm, mat in zip(names, mats):
    print(f"{nm:18s}: " + ", ".join(f"{k}={run(f, mat, strict=False)[1]}" for k, f in solvers.items()))

1 ở góc dưới-phải : linear=10000, binary=601, bisect=601, pruned=106, stair=101, colsearch=700
toàn 0            : linear=10000, binary=600, bisect=600, pruned=100, stair=100, colsearch=600
toàn 1            : linear=1, binary=700, bisect=700, pruned=8, stair=100, colsearch=7
ngẫu nhiên        : linear=183, binary=650, bisect=650, pruned=118, stair=199, colsearch=199


## 12. Bộ test

1. Ba ví dụ đề.
2. Edge: `1 × 1` (0 hoặc 1), một hàng, một cột, toàn 0, toàn 1, ma trận không vuông (`R ≠ C`).
3. Ngẫu nhiên (seed cố định) đúng ràng buộc `1..100`: đối chiếu với `truth(mat)` và kiểm tra **số lần gọi ≤ 1000** cho cách 2, 3, 4.
4. Kiểm tra cận trên lý thuyết: bậc thang luôn `≤ R + C − 1` lần gọi.

In [18]:
good = {"binary": brows, "bisect": bis, "pruned": bpr, "stair": stair, "colsearch": colsrch, "Solution": sol}
edge = [[[0]], [[1]], [[0, 0, 1, 1]], [[0], [0], [1]], [[0], [0], [0]],
        [[0, 0, 0], [0, 0, 0]], [[1, 1, 1], [1, 1, 1]], [[0, 0, 0, 0, 1], [0, 0, 1, 1, 1]],
        [[0, 1], [0, 0], [0, 0], [1, 1]]]
for mat in [m for m, _ in examples] + edge:
    for name, f in good.items():
        a, calls, st = run(f, mat)
        assert st == "OK" and a == truth(mat), (name, mat, a)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
rng = random.Random(7)
max_calls = {k: 0 for k in good}
for _ in range(300):
    R, Cc = rng.randint(1, 100), rng.randint(1, 100)
    mat = rand_mat(rng, R, Cc, p_empty=rng.random())
    want = truth(mat)
    for name, f in good.items():
        a, calls, st = run(f, mat)
        assert st == "OK" and a == want, (name, R, Cc)
        max_calls[name] = max(max_calls[name], calls)
    a, calls, st = run(stair, mat)
    assert calls <= R + Cc - 1
print("random 300 ma trận: OK | số lần gọi lớn nhất:", max_calls)

random 300 ma trận: OK | số lần gọi lớn nhất: {'binary': 639, 'bisect': 639, 'pruned': 129, 'stair': 185, 'colsearch': 588, 'Solution': 185}


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quét tuyến tính từng hàng (vượt giới hạn 1000 lần gọi)

In [20]:
class WrongScanRows:
    def leftMostColumnWithOne(self, binaryMatrix):
        rows, cols = binaryMatrix.dimensions()
        ans = cols
        for r in range(rows):
            for c in range(cols):            # SAI về số lần gọi
                if binaryMatrix.get(r, c) == 1:
                    ans = min(ans, c)
                    break
        return -1 if ans == cols else ans

bad = [[0] * 100 for _ in range(100)]
print("kết quả sai:", run(lambda bm: WrongScanRows().leftMostColumnWithOne(bm), bad))
print("đúng       :", run(sol, bad))

kết quả sai: (None, 1001, 'Wrong Answer (quá 1000 lần gọi get)')
đúng       : (-1, 100, 'OK')


**Vì sao sai:** đáp án đúng về giá trị nhưng gọi `get` tới 10 000 lần với ma trận 100 × 100 toàn 0 ⇒ LeetCode chấm **Wrong Answer** (bộ giả lập của ta ném `TooManyCalls` ở lần thứ 1001 và `run` bắt lại).

**Sửa:** binary search mỗi hàng hoặc đi bậc thang.

### Lỗi 2 — Bậc thang đi sai hướng (thấy 1 thì đi xuống)

In [21]:
class WrongDirection:
    def leftMostColumnWithOne(self, binaryMatrix):
        rows, cols = binaryMatrix.dimensions()
        r, c, ans = 0, cols - 1, -1
        while r < rows and c >= 0:
            if binaryMatrix.get(r, c) == 1:
                ans = c
                r += 1                       # SAI: phải sang trái
            else:
                c -= 1                       # SAI: phải xuống dưới
        return ans

wd = lambda bm: WrongDirection().leftMostColumnWithOne(bm)
for mat in [demo, [[0, 0], [0, 1]], [[0, 1], [1, 1]]]:
    print(mat, "| sai:", run(wd, mat)[0], "| đúng:", truth(mat))

[[0, 0, 0, 1, 1], [0, 0, 1, 1, 1], [0, 0, 0, 0, 0], [0, 1, 1, 1, 1]] | sai: 4 | đúng: 1
[[0, 0], [0, 1]] | sai: -1 | đúng: 1
[[0, 1], [1, 1]] | sai: 1 | đúng: 0


**Vì sao sai:** thấy `1` mà đi xuống thì bỏ lỡ khả năng có số 1 ở **bên trái** cùng hàng; thấy `0` mà đi trái thì vô ích (bên trái chắc chắn là 0). Với `[[0,1],[1,1]]` đáp án là 0 nhưng code sai trả 1.

**Sửa:** `1` ⇒ trái (`c −= 1`), `0` ⇒ xuống (`r += 1`).

### Lỗi 3 — Khởi tạo `ans = -1` rồi lấy `min`

In [22]:
class WrongInit:
    def leftMostColumnWithOne(self, binaryMatrix):
        rows, cols = binaryMatrix.dimensions()
        ans = -1                              # SAI: -1 luôn là min
        for r in range(rows):
            lo, hi = 0, cols
            while lo < hi:
                mid = (lo + hi) // 2
                if binaryMatrix.get(r, mid) == 1: hi = mid
                else: lo = mid + 1
            if lo < cols:
                ans = min(ans, lo)
        return ans

wi = lambda bm: WrongInit().leftMostColumnWithOne(bm)
for mat in [demo, [[0, 0], [1, 1]]]:
    print("sai:", run(wi, mat)[0], "| đúng:", truth(mat))

sai: -1 | đúng: 1
sai: -1 | đúng: 0


**Vì sao sai:** `min(-1, lo) = -1` với mọi `lo ≥ 0` ⇒ luôn trả `-1`. Giá trị "không có" dùng cho `min` phải là **lớn** (ví dụ `cols`).

**Sửa:** `ans = cols`, cuối cùng `return -1 if ans == cols else ans`.

### Lỗi 4 — Nhầm thứ tự `dimensions()` (`cols, rows`)

In [23]:
class WrongDims:
    def leftMostColumnWithOne(self, binaryMatrix):
        cols, rows = binaryMatrix.dimensions()     # SAI: trả về [rows, cols]
        r, c, ans = 0, cols - 1, -1
        while r < rows and c >= 0:
            try:
                v = binaryMatrix.get(r, c)
            except IndexError:
                return "IndexError!"               # bắt lại để in ra thay vì văng lỗi
            if v == 1:
                ans = c; c -= 1
            else:
                r += 1
        return ans

wdim = lambda bm: WrongDims().leftMostColumnWithOne(bm)
for mat in [[[0, 0, 0, 1], [0, 1, 1, 1]], [[0], [0], [1]]]:
    print(f"{len(mat)}x{len(mat[0])}", "| sai:", run(wdim, mat)[0], "| đúng:", truth(mat))

2x4 | sai: IndexError! | đúng: 1
3x1 | sai: IndexError! | đúng: 0


**Vì sao sai:** `dimensions()` trả `[rows, cols]`. Đảo thứ tự vẫn chạy đúng với ma trận **vuông** (nên dễ lọt qua ví dụ đề), nhưng với ma trận chữ nhật thì truy cập ngoài biên hoặc bỏ sót cột.

**Sửa:** `rows, cols = binaryMatrix.dimensions()` và luôn test ma trận không vuông.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Ma trận sắp theo hàng (và/hoặc cột)** ⇒ nghĩ ngay đến **đi bậc thang** từ một góc mà hai hướng đi có ý nghĩa ngược nhau (trên-phải hoặc dưới-trái). Giống **LeetCode 240 Search a 2D Matrix II**, **1351 Count Negative Numbers in a Sorted Matrix**.
- Bài **tương tác có giới hạn số lần hỏi** ⇒ tính trước số lần hỏi tệ nhất: `R·C = 10^4` ✘, `R·log C = 700` ✔, `R + C = 200` ✔.
- Dãy `0…0 1…1` ⇒ binary search "vị trí đầu tiên thoả" — học thuộc khuôn `lo < hi`, `hi = mid`, `lo = mid + 1`.
- Vị từ đơn điệu theo đáp án ⇒ **binary search on answer**.
- Luôn test: toàn 0 (`-1`), toàn 1 (`0`), không vuông, `1 × 1`.

## 15. Biến thể của bài

1. **Ma trận sắp theo cột** (mỗi cột 0…0 1…1 từ trên xuống): tìm **hàng trên cùng** có số 1 ⇒ bậc thang từ góc **dưới-trái**.
2. **Hàng có nhiều số 1 nhất** (LeetCode 1337 The K Weakest Rows là bản ngược): cùng bậc thang, ghi lại hàng làm `c` giảm.
3. **Đếm số 1** trong ma trận sắp cả hàng lẫn cột ⇒ bậc thang `O(R + C)` (như 1351).
4. **LeetCode 240**: tìm `target` trong ma trận sắp theo hàng và cột ⇒ bậc thang từ góc trên-phải.
5. **LeetCode 278 First Bad Version / 374 Guess Number**: bài tương tác binary search một chiều.

Code biến thể 4 (LeetCode 240, minh hoạ cùng kỹ thuật bậc thang):

In [24]:
def search_matrix_240(mat, target):
    r, c = 0, len(mat[0]) - 1
    while r < len(mat) and c >= 0:
        if mat[r][c] == target:
            return True
        if mat[r][c] > target:
            c -= 1
        else:
            r += 1
    return False

m240 = [[1, 4, 7, 11], [2, 5, 8, 12], [3, 6, 9, 16], [10, 13, 14, 17]]
print(search_matrix_240(m240, 5), search_matrix_240(m240, 15))

True False


## 16. So sánh các cách

| Cách | Ý tưởng | Số lần gọi | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét tuyến tính | Duyệt từng ô đến khi gặp 1 | `R·C` (✘ > 1000) | `O(1)` | Chỉ để hiểu bài |
| 2. Binary search mỗi hàng | Điểm chuyển 0→1 của từng hàng, lấy min | `R·log C` ≤ 700 | `O(1)` | Dễ nghĩ, dùng `bisect` |
| 2b. Có tỉa theo `ans` | Chỉ tìm trong `[0, ans)` | ≤ 700, thường ít hơn | `O(1)` | Cải tiến nhanh |
| 3. Bậc thang | Góc trên-phải: 1 ⇒ trái, 0 ⇒ xuống | `R + C − 1` ≤ 199 | `O(1)` | Tối ưu, nên nộp |
| 4. Binary search trên cột | Vị từ "cột có 1" đơn điệu | `R·log C` ≤ 700 | `O(1)` | Luyện "binary search on answer" |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Với `mat = [[0,0,0,1],[0,0,1,1],[0,0,0,0]]`, `Solution` trả gì và gọi `get` bao nhiêu lần?

### Bài 2 (tính tay)
Đi bậc thang tay trên `mat = [[0,0,0,1,1],[0,0,1,1,1],[0,0,0,0,0],[0,1,1,1,1]]` (ma trận `demo`): liệt kê các ô được hỏi và đếm số lần gọi.

### Bài 3 (điền TODO)
Hoàn thiện `first_one(bm, r, cols)`: binary search trả vị trí số 1 đầu tiên của hàng `r`, hoặc `cols` nếu không có.

### Bài 4 (tự viết lại)
Viết `leftmost_with_row(bm)`: đi bậc thang và trả về cặp `(cột, hàng)` — cột nhỏ nhất có 1 và **hàng nhỏ nhất** có số 1 ở cột đó; `(-1, -1)` nếu không có.

### Bài 5 (mini challenge)
Ma trận **sắp theo cột** (mỗi cột 0…0 rồi 1…1 từ trên xuống), giao diện giống `BinaryMatrix`. Viết `topmost_row_with_one(bm)` trả hàng nhỏ nhất có chứa số 1 (hoặc `-1`), dùng `≤ R + C` lần gọi.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def first_one(bm, r, cols):
    lo, hi = 0, cols
    # TODO: vòng while lo < hi với mid, dùng bm.get(r, mid)
    pass

# Bài 4 — TODO
def leftmost_with_row(bm):
    # TODO: bậc thang; khi thấy 1 thì ghi (c, r) rồi sang trái
    pass

# Bài 5 — TODO
def topmost_row_with_one(bm):
    # TODO: bắt đầu ở góc dưới-trái (R-1, 0); thấy 1 -> đi lên, thấy 0 -> sang phải
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Bắt đầu `(0, 3) = 1` ⇒ trái. (2) `(0, 2) = 0` ⇒ xuống; `(1, 2) = 1` ⇒ trái; `(1, 1) = 0` ⇒ xuống; `(2, 1) = 0` ⇒ xuống, `r = 3` dừng. (3) Kết quả `2`, **5** lần gọi.

**Bài 2.** (1) Xem bảng ở mục 8. (2) Các ô: `(0,4) (0,3) (0,2) (1,2) (1,1) (2,1) (3,1) (3,0)`. (3) 8 lần gọi, kết quả `1`.

**Bài 3.** (1) `mid = (lo + hi) // 2`. (2) `get == 1` ⇒ `hi = mid`, ngược lại `lo = mid + 1`. (3) Trả `lo`.

**Bài 4.** (1) Biến `best = (-1, -1)`. (2) Thấy 1 tại `(r, c)` ⇒ `best = (c, r)`. (3) Hàng ghi lại là hàng đầu tiên (từ trên) có 1 ở cột cuối cùng — vì các hàng trước đã có 0 ở cột đó.

**Bài 5.** (1) Chuyển vai trò hàng ↔ cột so với bài gốc. (2) Ở `(r, c)`: 1 ⇒ hàng `r` có 1, ghi `ans = r`, đi lên `r −= 1`; 0 ⇒ cả cột `c` từ hàng `r` lên trên đều 0, sang phải `c += 1`. (3) Dừng khi `r < 0` hoặc `c == cols`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
a, calls, st = run(sol, [[0, 0, 0, 1], [0, 0, 1, 1], [0, 0, 0, 0]])
assert (a, calls, st) == (2, 5, "OK")

# Bài 2
a, calls, st = run(sol, demo)
assert (a, calls) == (1, 8)

# Bài 3
def first_one_sol(bm, r, cols):
    lo, hi = 0, cols
    while lo < hi:
        mid = (lo + hi) // 2
        if bm.get(r, mid) == 1:
            hi = mid
        else:
            lo = mid + 1
    return lo

# Bài 4
def leftmost_with_row_sol(bm):
    rows, cols = bm.dimensions()
    r, c, best = 0, cols - 1, (-1, -1)
    while r < rows and c >= 0:
        if bm.get(r, c) == 1:
            best = (c, r)
            c -= 1
        else:
            r += 1
    return best

# Bài 5
def topmost_row_with_one_sol(bm):
    rows, cols = bm.dimensions()
    r, c, ans = rows - 1, 0, -1
    while r >= 0 and c < cols:
        if bm.get(r, c) == 1:
            ans = r
            r -= 1
        else:
            c += 1
    return ans

rng = random.Random(11)
for _ in range(300):
    R, Cc = rng.randint(1, 30), rng.randint(1, 30)
    mat = rand_mat(rng, R, Cc, p_empty=rng.random())
    bm = BinaryMatrix(mat)
    for r in range(R):
        assert first_one_sol(bm, r, Cc) == (mat[r].index(1) if 1 in mat[r] else Cc)
    col = truth(mat)
    want_row = -1 if col == -1 else min(r for r in range(R) if mat[r][col] == 1)
    assert leftmost_with_row_sol(BinaryMatrix(mat)) == (col, want_row)
    # ma trận sắp theo cột = chuyển vị của ma trận sắp theo hàng
    tmat = [list(x) for x in zip(*mat)]
    bm2 = BinaryMatrix(tmat)
    top = topmost_row_with_one_sol(bm2)
    rows_with_one = [r for r in range(len(tmat)) if 1 in tmat[r]]
    assert top == (min(rows_with_one) if rows_with_one else -1)
    assert bm2.calls <= len(tmat) + len(tmat[0])
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template bậc thang (ma trận sắp theo hàng, tìm cột trái nhất có 1):**

```python
rows, cols = binaryMatrix.dimensions()
r, c, ans = 0, cols - 1, -1
while r < rows and c >= 0:
    if binaryMatrix.get(r, c) == 1:
        ans, c = c, c - 1       # có 1: ghi nhận, sang trái
    else:
        r += 1                  # 0: hàng này hết hy vọng
return ans
```

**Template binary search "vị trí đầu tiên thoả":**

```python
lo, hi = 0, n
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid
    else: lo = mid + 1
return lo        # n nếu không có
```

**Checklist:**
- [ ] Đã tính số lần gọi `get` tệ nhất ≤ 1000.
- [ ] `rows, cols = dimensions()` đúng thứ tự.
- [ ] 1 ⇒ sang trái, 0 ⇒ xuống dưới.
- [ ] Giá trị khởi tạo cho `min` là `cols`, không phải `-1`.
- [ ] Trả `-1` khi không có số 1.
- [ ] Đã test toàn 0, toàn 1, ma trận không vuông, `1 × 1` (bằng bộ giả lập `BinaryMatrix`).